# Paso 1

#### Librerias y lectura

In [1]:
import json
import re
import pandas as pd
import unicodedata
from thefuzz import fuzz

In [2]:
df = pd.read_json('catalogo_tp3.jsonl', lines=True)

#### Vista inicial

In [3]:
total_inicial = len(df)
print(f"Total de registros iniciales: {total_inicial}")

Total de registros iniciales: 8747


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8747 entries, 0 to 8746
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   source_url      8747 non-null   str                
 1   slug            8747 non-null   str                
 2   title           8747 non-null   str                
 3   authors         8747 non-null   object             
 4   genres          8747 non-null   object             
 5   seed_genre      8747 non-null   str                
 6   series          2316 non-null   str                
 7   series_number   2298 non-null   float64            
 8   cover_url       8747 non-null   str                
 9   synopsis        8746 non-null   str                
 10  synopsis_chars  8747 non-null   int64              
 11  scraped_at      8747 non-null   datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), float64(1), int64(1), object(2), str(7)
memory usage: 820.2+ KB


In [5]:
df.head()

,source_url,slug,title,authors,genres,seed_genre,series,series_number,cover_url,synopsis,synopsis_chars,scraped_at
0,https://ww3.lectulandia.com/book/de-la-caverna...,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],"[Arqueología, Divulgación, Ensayo]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eudald%20C...,La Humanidad ha entrado en una nueva fase evol...,998,2026-10-17 23:11:56+00:00
1,https://ww3.lectulandia.com/book/quienes-somos...,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],"[Arqueología, Biología, Ciencias naturales, En...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/David%20Re...,La extracción de ADN de los huesos de nuestros...,1285,2026-10-17 23:11:58+00:00
2,https://ww3.lectulandia.com/book/teotihuacan/,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],"[Arqueología, Arte, Ciencias sociales, Ensayo,...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Eduardo%20...,Teotihuacan es la primera gran ciudad del cent...,1070,2026-10-17 23:12:01+00:00
3,https://ww3.lectulandia.com/book/tiro-y-las-co...,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],"[Arqueología, Economía, Ensayo, Historia, Polí...",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/M%20Eugeni...,Entre los siglos VIII y VI a. C. los fenicios ...,1875,2026-10-17 23:12:03+00:00
4,https://ww3.lectulandia.com/book/la-maldicion-...,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],"[Arqueología, Ensayo, Psicología, Sociología]",arqueologia,NaN,NaN,https://assets.lectulandia.com/b/ab/Emiliano%2...,Las raíces evolutivas del sufrimiento humano\n...,1613,2026-10-17 23:12:05+00:00


#### Analisis de sinopsis

In [6]:
(df['synopsis'] == "").sum()

np.int64(0)

No hay sinopsis vacias

In [ ]:
df[df['synopsis'].isnull()]

,source_url,slug,title,authors,genres,seed_genre,series,series_number,cover_url,synopsis,synopsis_chars,scraped_at
7969,https://ww3.lectulandia.com/book/calvario/,calvario,Calvario,[Alejandro Sawa],"[Drama, Teatro]",teatro,NaN,NaN,https://assets.lectulandia.com/b/ab/Alejandro%...,NaN,0,2026-10-18 22:48:58+00:00


Hay una sola fila con sinapsis vacia

In [8]:
df = df.dropna(subset=["synopsis"])
df = df[df["synopsis"].astype(str).str.strip() != ""]

In [ ]:
df.duplicated(subset=['slug']).sum()

np.int64(0)

No hay libros duplicados

In [10]:
df = df.drop_duplicates(subset=["slug"], keep="first")

In [11]:
len(df)

8746

In [ ]:
# Normalizar espacios en blanco (reemplazar saltos de línea y tabs por un único espacio)
antes = len(df)
df["synopsis"] = df["synopsis"].astype(str).apply(lambda x: re.sub(r"\s+", " ", x).strip())
df["title"] = df["title"].astype(str).apply(lambda x: re.sub(r"\s+", " ", x).strip())

In [13]:
# Contar duplicados exactos de sinopsis
sinopsis_duplicadas = df[df.duplicated(subset=["synopsis"], keep=False)]
print(f"Registros con sinopsis iguales: {len(sinopsis_duplicadas)}")

Registros con sinopsis iguales: 131


In [14]:
# Conservar una sola versión por sinopsis
df = df.drop_duplicates(subset=["synopsis"], keep="first")
eliminados_sinopsis = antes - len(df)
print(f"Libros eliminados por sinopsis iguales: {eliminados_sinopsis}")

Libros eliminados por sinopsis iguales: 98


#### Busqueda de libros "casi iguales"

Normalizacion de los titulos

In [ ]:
def normalizar_titulo(texto):
    if pd.isna(texto):
        return ""

    texto = texto.lower()

    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )

    texto = re.sub(r"[^\w\s]", " ", texto)

    texto = re.sub(r"\s+", " ", texto).strip()

    return texto


df["title_norm"] = df["title"].apply(normalizar_titulo)

Buscar candidatos

In [ ]:
df["authors_norm"] = df["authors"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else str(x)
)

pares = []

romanos = r"\b(i|ii|iii|iv|v|vi|vii|viii|ix|x)\b"

for autor, grupo in df.groupby("authors_norm"):

    grupo = grupo.reset_index()

    if len(grupo) < 2:
        continue

    for i in range(len(grupo)):
        for j in range(i + 1, len(grupo)):

            titulo1 = grupo.loc[i, "title_norm"]
            titulo2 = grupo.loc[j, "title_norm"]

            score = fuzz.token_set_ratio(titulo1, titulo2)

            if score < 90:
                continue

            # Si tienen números distintos (años, tomos, etc.)
            nums1 = re.findall(r"\d+", titulo1)
            nums2 = re.findall(r"\d+", titulo2)

            if nums1 != nums2:
                continue

            # Si cambian los números romanos
            rom1 = re.findall(romanos, titulo1)
            rom2 = re.findall(romanos, titulo2)

            if rom1 != rom2:
                continue

            # Evitar tomos o volúmenes diferentes
            palabras = ["vol", "volumen", "tomo", "parte", "libro", "numero"]

            if any(p in titulo1 or p in titulo2 for p in palabras):
                if titulo1 != titulo2:
                    continue

            pares.append({
                "idx1": grupo.loc[i, "index"],
                "idx2": grupo.loc[j, "index"],
                "autor": autor,
                "titulo1": grupo.loc[i, "title"],
                "titulo2": grupo.loc[j, "title"],
                "score_titulo": score
            })

pares = pd.DataFrame(pares)

print(f"Pares candidatos encontrados: {len(pares)}")
display(pares.head())

Pares candidatos encontrados: 110


,idx1,idx2,autor,titulo1,titulo2,score_titulo
0,451,7950,"Adolfo Bioy Casares, Jorge Luis Borges",Nuevos cuentos de Bustos Domecq,Nuevos cuentos de Bustos Domecq,100
1,490,2090,Alejandro Fabbri,Historias negras del fútbol argentino,Nuevas historias negras del fútbol argentino,100
2,529,2857,Ambrose Bierce,El diccionario del Diablo (Trad. Eduardo Stilman),El diccionario del Diablo,100
3,2857,3590,Ambrose Bierce,El diccionario del Diablo,El diccionario del diablo (trad. Rodolfo Walsh),100
4,1603,7955,Anónimo,Lazarillo de Tormes (ed. Francisco Rico),Lazarillo de Tormes,100


Comparar sinopsis

In [17]:
pares["score_sinopsis"] = pares.apply(
    lambda fila: fuzz.token_set_ratio(
        df.loc[fila["idx1"], "synopsis"],
        df.loc[fila["idx2"], "synopsis"]
    ),
    axis=1
)

display(pares.head())

,idx1,idx2,autor,titulo1,titulo2,score_titulo,score_sinopsis
0,451,7950,"Adolfo Bioy Casares, Jorge Luis Borges",Nuevos cuentos de Bustos Domecq,Nuevos cuentos de Bustos Domecq,100,59
1,490,2090,Alejandro Fabbri,Historias negras del fútbol argentino,Nuevas historias negras del fútbol argentino,100,58
2,529,2857,Ambrose Bierce,El diccionario del Diablo (Trad. Eduardo Stilman),El diccionario del Diablo,100,51
3,2857,3590,Ambrose Bierce,El diccionario del Diablo,El diccionario del diablo (trad. Rodolfo Walsh),100,58
4,1603,7955,Anónimo,Lazarillo de Tormes (ed. Francisco Rico),Lazarillo de Tormes,100,59


Elegir las posibles "obras iguales" y mostrarlas

In [18]:
obras_iguales = pares[
    pares["score_sinopsis"] >= 80
].sort_values(
    ["score_titulo", "score_sinopsis"],
    ascending=False
)

print(f"Posibles obras iguales: {len(obras_iguales)}")

Posibles obras iguales: 25


Mostramos estas obras

In [19]:
display(
    obras_iguales[
        [
            "idx1",
            "idx2",
            "autor",
            "titulo1",
            "titulo2",
            "score_titulo",
            "score_sinopsis"
        ]
    ]
)

,idx1,idx2,autor,titulo1,titulo2,score_titulo,score_sinopsis
7,1309,1504,Arthur Conan Doyle,Sherlock Holmes,Sherlock Holmes. La colección completa,100,100
37,4862,5876,H. Rider Haggard,"Ella, Hija de la Sabiduría",Ella. Hija de la sabiduría,100,100
43,206,6001,Isaac Asimov,El secreto del universo,El secreto del universo y otros ensayos cientí...,100,100
53,208,6218,John Gribbin,Introducción a la ciencia. Una guía para todos...,Introducción a la ciencia,100,100
61,1550,1803,Lewis Wallace,Ben-Hur,Ben-Hur (Traducción Raimundo Griñó),100,100
64,2065,8462,Mark Twain,El príncipe y el mendigo,Príncipe y mendigo,100,100
65,736,739,Maurice Nicoll,Comentarios psicológicos sobre las enseñanzas ...,Comentarios psicológicos sobre las enseñanzas ...,100,100
82,8028,8032,Richard Ford,Manual para viajeros por Andalucía y lectores ...,Manual para viajeros por Andalucía y lectores ...,100,100
49,359,7456,Jean-Jacques Rousseau,El contrato social,Contrato social,100,99
60,801,5290,Laura Gallego García,La emperatriz de los Etéreos (Ed. Ilustrada),La emperatriz de los Etéreos,100,99


### Decisión sobre los candidatos a obras iguales

La detección automática produjo 25 pares candidatos a corresponder a una misma obra. Sin embargo, la similitud de títulos y sinopsis no garantiza que dos registros representen el mismo libro.

Tras una revisión manual, solo se eliminaron aquellos casos en los que existía evidencia suficiente de que ambos registros correspondían a la misma obra, diferenciándose únicamente por aspectos editoriales como puntuación, subtítulos, traducciones o la indicación de una edición ilustrada.

En cambio, se conservaron aquellos pares que, aunque presentan una alta similitud textual, representan obras distintas. Entre ellos se encuentran recopilaciones que contienen material adicional (por ejemplo, *Sherlock Holmes. La colección completa* o *El secreto del universo y otros ensayos científicos*), así como libros pertenecientes a una misma colección pero dedicados a regiones diferentes (por ejemplo, las guías de viaje de Richard Ford para Andalucía, Castilla, Cataluña o el País Vasco). Eliminar estos registros implicaría perder información válida del catálogo.

Por este motivo, la decisión final combina la detección automática de candidatos con una validación manual, priorizando la conservación de información cuando existen dudas sobre si ambos registros corresponden realmente a la misma obra.

In [ ]:
ids_eliminar = [
    5876,  # Ella. Hija de la sabiduría (diferencia solo de puntuación)
    1803,  # Ben-Hur (Traducción Raimundo Grinó)
    8462,  # Príncipe y mendigo
    739,   # Comentarios psicológicos sobre las enseñanzas...
    8032,  # Manual para viajeros por Andalucía y lectores... (duplicado exacto)
    7456,  # Contrato social
    8733,  # El hombre que fue Jueves (trad. José Rafael...)
    5290   # La emperatriz de los Etéreos (Ed. Ilustrada)
]

df = df.drop(index=ids_eliminar).reset_index(drop=True)

print(f"Registros eliminados: {len(ids_eliminar)}")
print(f"Total de libros luego de la limpieza: {len(df)}")

Registros eliminados: 8
Total de libros luego de la limpieza: 8640


# PASO 2

## 2.1 Expresiones regulares

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, confusion_matrix

df_muestra = pd.read_csv("muestra_etiquetada.csv", keep_default_na=False)
desarrollo, prueba = train_test_split(df_muestra, test_size=0.3, random_state=42, stratify=df_muestra["contaminado"])

print(f"Tamaño del conjunto de desarrollo: {len(desarrollo)}")
print(f"Tamaño del conjunto de prueba: {len(prueba)}")

def medir(nombre, marcado, datos):
    y = datos["contaminado"]
    p = precision_score(y, marcado, zero_division=0)
    r = recall_score(y, marcado)
    print(f"{nombre}: precisión {p:.2f}, recall {r:.2f}")
    return p, r

Tamaño del conjunto de desarrollo: 350
Tamaño del conjunto de prueba: 150


In [22]:
df_muestra.head()

,slug,title,synopsis,contaminado,fragmento
0,el-cinturon-de-venus,El cinturón de Venus,"La parábola, la intriga y el humor, tan escaso...",0,
1,los-hijos-de-la-llanura-2,Los hijos de la llanura,Una historia apasionante que te atrapará desde...,1,Una historia apasionante que te atrapará desde...
2,electra,Electra,Entre las siete tragedias de Sófocles (c. 496 ...,0,
3,el-despertar-de-la-senorita-prim,El despertar de la señorita Prim,Atraída por un sugestivo anuncio en un periódi...,0,
4,comprometer-a-un-vizconde,Comprometer a un Vizconde,"Su vida es el ejército, no tiene tiempo para e...",0,


In [23]:
# Regex para URLs, correos y teléfonos
URL = r"https?://\S+|www\.\S+"
# Dominio escrito sin http ni www, por ejemplo "librohub.test"
DOMINIO = r"\b[a-z0-9-]{3,}\.(?:com|net|org|info|es|ar|mx|io|test|example|invalid)\b(?:/\S*)?"
CORREO = r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"
# Teléfono con prefijo internacional (+54 9 11 5217-8319) o en tres grupos de dígitos.
# Una versión más laxa (\+?\d[\d\s\-\(\)]{7,15}\d) marcaba rangos de años como "1914-1918",
# muy comunes en las sinopsis de historia: 265 libros del catálogo.
TELEFONO = r"\+\d{1,3}[\s-]?(?:\d[\s-]?){8,12}\d|\b\d{3,4}[\s-]\d{3,4}[\s-]\d{4}\b"

patron_contaminacion = re.compile(f"{URL}|{DOMINIO}|{CORREO}|{TELEFONO}", re.IGNORECASE)

def predecir_con_regex(texto):
    return int(bool(patron_contaminacion.search(str(texto))))

def coincidencia(texto):
    m = patron_contaminacion.search(str(texto))
    return m.group(0) if m else ""

y_pred_regex = prueba["synopsis"].apply(predecir_con_regex)
medir("regex", y_pred_regex, prueba)

regex: precisión 1.00, recall 0.40


(1.0, 0.4)

#### Revisión de errores

In [24]:
# Falsos positivos en toda la muestra y en el catálogo: qué fragmento hizo saltar la regex
marcado_muestra = df_muestra["synopsis"].apply(predecir_con_regex)
fp_muestra = df_muestra[(marcado_muestra == 1) & (df_muestra["contaminado"] == 0)]
print(f"Falsos positivos en la muestra: {len(fp_muestra)}")
for texto in fp_muestra["synopsis"]:
    print(f"  {coincidencia(texto)!r:40} | {texto[:90]}")

regex_catalogo = df["synopsis"].apply(coincidencia)
print(f"\nLibros del catálogo marcados por la regex: {(regex_catalogo != '').sum()}")
print("Dominios reales que aparecen en sinopsis del catálogo (posibles falsos positivos):")
print(regex_catalogo[regex_catalogo.str.contains(r"\.(?:com|es|org|net|ar)\b", case=False)].head(15).tolist())

Falsos positivos en la muestra: 2
  'Diverso.es'                             | Astur-madrileño de 1969, Adolfo Cueto se dio a conocer poéticamente con la aparición de Di
  'http://www.xkcd.com'                    | Un libro en el que el humor y la ciencia se unen para dar explicación a todo tipo de pregu



Libros del catálogo marcados por la regex: 131
Dominios reales que aparecen en sinopsis del catálogo (posibles falsos positivos):
['wordpress.com', 'www.perezreverte.com', 'Retosmentales.com', 'http://www.xkcd.com', 'elmus.org', 'tusescritos.com', 'eldiario.es', 'Diverso.es', 'Monstruopedia.com', '20minutos.es', 'escritoras.com', 'www.perezreverte.com', 'wordpress.com', 'www.depechemode.com']


In [25]:
# Falsos negativos en prueba: texto insertado que la regex no encontró
fn = prueba[(y_pred_regex == 0) & (prueba["contaminado"] == 1)]
for f in fn["fragmento"]:
    print("-", f[:130])

- Ganá dinero desde tu casa recomendando libros. Cupos limitados, consultá requisitos.
- Una historia apasionante que te atrapará desde la primera página hasta el final. Ganá dinero desde tu casa recomendando libros. Cu
- Entrá a tоmoclub.example y leé sin límites.
- Ganá dinero desde tu casa recomendando libros. Cupos limitados, consultá requisitos.
- ¡Oferta por tiempo limitado! Suscribite hoy y llevate tres libros de regalo.
- Buscalo en w w w . novelalibre . example
- Si te gustó esta reseña, compartila con tus amigos y participá del sorteo de un lector electrónico.
- Buscalo en w w w . letramax . example
- Entrá a letrаya.invalid y leé sin límites.
- Link en nuestro sitio: h t t p s : / / lectortotal [.] example
- Leia milhares de livros sem pagar nada com a nossa oferta por tempo limitado!
- Get unlimited access to thousands of bestsellers with our limited time offer!
- Descargalo gratis en TomoMax punto example barra cardamomo
- Leia milhares de livros sem pagar nada com a nossa

#### Respuesta 2.1

La regex detecta bien la contaminación **explícita**: enlaces con `http`/`www`, dominios escritos completos (`librohub.test`), correos (`ventas@...`) y teléfonos con prefijo (`+54 9 11 ...`). En prueba tiene precisión alta pero recall bajo (≈0.40): no detecta los enlaces **disimulados** (letras separadas, `[.]`, «punto», caracteres invisibles o cirílicos) ni el **texto promocional sin enlace** («Conseguí este título y más de 50.000 libros en la app...»).

Los falsos positivos son sinopsis legítimas que mencionan sitios reales. En la muestra aparecen `Diverso.es` (el nombre de un poemario) y `http://www.xkcd.com`; en el catálogo, `eldiario.es`, `20minutos.es`, `wordpress.com` o la página de Pérez-Reverte. La primera versión de la regex de teléfonos además marcaba rangos de años como `1914-1918`: se la restringió a números con prefijo internacional o en tres grupos.

## 2.2 Normalizar antes de buscar

In [ ]:
from collections import Counter

raros = Counter()
for texto in desarrollo["synopsis"]:
    for c in texto:
        if unicodedata.category(c) == "Cf" or (c.isalpha() and ord(c) > 0x024F):
            raros[(c, unicodedata.name(c, "?"), unicodedata.category(c))] += 1
display(pd.DataFrame([(repr(c), n, cat, k) for (c, n, cat), k in raros.most_common(20)],
                     columns=["carácter", "nombre", "categoría", "veces"]))

,carácter,nombre,categoría,veces
0,'\u2060',WORD JOINER,Cf,9
1,'\xad',SOFT HYPHEN,Cf,5
2,'\u200b',ZERO WIDTH SPACE,Cf,4
3,'а',CYRILLIC SMALL LETTER A,Ll,2
4,'і',CYRILLIC SMALL LETTER BYELORUSSIAN-UKRAINIAN I,Ll,2
5,'е',CYRILLIC SMALL LETTER IE,Ll,2
6,'о',CYRILLIC SMALL LETTER O,Ll,1
7,'с',CYRILLIC SMALL LETTER ES,Ll,1


In [ ]:
# Letras cirílicas y griegas que se ven iguales a las latinas
HOMOGLIFOS = str.maketrans({
    "а": "a", "е": "e", "о": "o", "р": "p", "с": "c", "у": "y", "х": "x", "і": "i", "ј": "j", "ѕ": "s",
    "А": "A", "Е": "E", "О": "O", "Р": "P", "С": "C", "Т": "T", "Н": "H", "К": "K", "М": "M", "В": "B", "Х": "X",
    "ο": "o", "α": "a", "ε": "e", "ι": "i", "κ": "k", "ν": "v", "ρ": "p", "τ": "t", "υ": "u",
    "Ο": "O", "Α": "A", "Ε": "E", "Ι": "I", "Κ": "K", "Ν": "N", "Ρ": "P", "Τ": "T",
})
TLD = r"(?:com|net|org|test|example|invalid)"

def normalizar(texto):
    # Formas de ancho completo y similares (ｗ -> w)
    texto = unicodedata.normalize("NFKC", str(texto))
    # Caracteres invisibles (categoría Cf: espacio de ancho cero, unión de palabras, etc.)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Cf")
    # Letras de otro alfabeto que se ven como latinas
    texto = texto.translate(HOMOGLIFOS)
    # Letras separadas por espacios ("h t t p s", "w w w")
    texto = re.sub(r"\b(?:\w ){3,}\w\b", lambda m: m.group(0).replace(" ", ""), texto)
    # Signos escritos con espacios o con palabras
    texto = re.sub(r"\s*:\s*/\s*/\s*", "://", texto)
    texto = re.sub(r"\bwww\s*\.\s*", "www.", texto, flags=re.IGNORECASE)
    texto = re.sub(rf"\s*\.\s+(?={TLD}\b)", ".", texto)
    texto = re.sub(r"\s*[\[(]\s*(?:\.|dot|punto)\s*[\])]\s*", ".", texto, flags=re.IGNORECASE)
    texto = re.sub(rf"\s+(?:punto|dot)\s+(?={TLD}\b)", ".", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\s+arroba\s+", "@", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\s+barra\s+", "/", texto, flags=re.IGNORECASE)
    return texto

def predecir_normalizado(texto):
    return predecir_con_regex(normalizar(texto))

y_pred_norm = prueba["synopsis"].apply(predecir_normalizado)
medir("regex sobre texto normalizado", y_pred_norm, prueba)

regex sobre texto normalizado: precisión 1.00, recall 0.63


(1.0, 0.6333333333333333)

In [ ]:
# Casos que la normalización recuperó 
recuperados = df_muestra[(df_muestra["synopsis"].apply(predecir_con_regex) == 0) &
                         (df_muestra["synopsis"].apply(predecir_normalizado) == 1) &
                         (df_muestra["contaminado"] == 1)]
print(f"Contaminados recuperados por la normalización: {len(recuperados)}\n")

for frag in recuperados["fragmento"].drop_duplicates().head(4):
    print(f"Original   : {frag!r}")
    print(f"Normalizado: {normalizar(frag)!r}")
    print(f"Coincidencia: {coincidencia(normalizar(frag))}\n")

Contaminados recuperados por la normalización: 20

Original   : 'Descargalo gratis en TomoMax punto example barra cardamomo'
Normalizado: 'Descargalo gratis en TomoMax.example/cardamomo'
Coincidencia: TomoMax.example/cardamomo

Original   : 'Entrá a lесtоrflash.example y leé sin límites.'
Normalizado: 'Entrá a lectorflash.example y leé sin límites.'
Coincidencia: lectorflash.example

Original   : 'El libro más esperado del año llega por fin en edición digital completa. Descargalo gratis en LibroYa punto test barra sombras-sobre-el-desierto'
Normalizado: 'El libro más esperado del año llega por fin en edición digital completa. Descargalo gratis en LibroYa.test/sombras-sobre-el-desierto'
Coincidencia: LibroYa.test/sombras-sobre-el-desierto

Original   : 'Entrá a librоexpress.invalid y leé sin límites.'
Normalizado: 'Entrá a libroexpress.invalid y leé sin límites.'
Coincidencia: libroexpress.invalid



#### Respuesta 2.2

La tabla de caracteres muestra que en la muestra hay espacios de ancho cero (`ZERO WIDTH SPACE`, `WORD JOINER`, categoría `Cf`) y letras cirílicas (`CYRILLIC SMALL LETTER O`, `A`, `IE`...) que se ven iguales a las latinas. `normalizar()` aplica NFKC, elimina los caracteres `Cf`, reemplaza los homoglifos cirílicos y griegos, junta letras separadas por espacios y reescribe signos disimulados («punto», `[.]`, `(punto)`, «arroba», «barra»).

Con la misma regex sobre el texto normalizado el recall en prueba sube (≈0.40 → ≈0.63) sin agregar falsos positivos. Los casos recuperados de arriba muestran, por ejemplo, `h t t p s : / / tomoya [.] test` → `https://tomoya.test` y `tоmoclub.example` (con «о» cirílica) → `tomoclub.example`.

Los caracteres invisibles también aparecen en sinopsis reales, así que por sí solos no se usan como evidencia: solo se eliminan para que la regex pueda encontrar el enlace. Por la misma razón, el punto se pega solo delante de terminaciones de dominio (`com`, `test`, `example`...), no en cualquier lugar del texto: pegar todos los puntos producía falsos positivos como `historia.Es`.

## 2.3 Detectar el idioma de cada oración

In [29]:
from langdetect import detect, detect_langs, DetectorFactory
DetectorFactory.seed = 42

def detectar_idioma_global(texto):
    try:
        return detect(str(texto))
    except Exception:
        return "unknown"

idiomas_globales = desarrollo["synopsis"].apply(detectar_idioma_global)
print(pd.crosstab(idiomas_globales, desarrollo["contaminado"]))

contaminado    0   1
synopsis            
es           280  70


Con la sinopsis completa, `langdetect` devuelve `es` para **todas** las sinopsis de desarrollo, contaminadas o no: el mensaje insertado en otro idioma es una o dos oraciones dentro de un texto mayormente en castellano, y el idioma dominante lo tapa. Por eso hay que detectar el idioma oración por oración.

In [30]:
def oraciones(texto):
    partes = re.split(r"(?<=[.!?])\s+|\n", texto)
    return [p.strip() for p in partes if len(p.strip()) >= 15]

def info_oraciones(texto):
    """Para cada oración: (cantidad de palabras, idioma más probable, su probabilidad)."""
    out = []
    for o in oraciones(texto):
        try:
            l = detect_langs(o)[0]
            out.append((len(o.split()), l.lang, l.prob))
        except Exception:
            pass
    return out

def marcar(info, min_pal, umbral):
    return int(any(n >= min_pal and lang != "es" and p >= umbral for n, lang, p in info))

In [31]:
info_dev = desarrollo["synopsis"].apply(info_oraciones)

filas = []
for mp in [3, 4, 5, 6, 8, 10]:
    for um in [0.7, 0.8, 0.9, 0.95, 0.99]:
        pred = info_dev.apply(lambda i: marcar(i, mp, um))
        filas.append((mp, um,
                      precision_score(desarrollo["contaminado"], pred, zero_division=0),
                      recall_score(desarrollo["contaminado"], pred)))

tabla = pd.DataFrame(filas, columns=["min_pal", "umbral", "precision", "recall"])
display(tabla.sort_values(["precision", "recall"], ascending=False).head(15))

,min_pal,umbral,precision,recall
22,8,0.90,0.818182,0.128571
23,8,0.95,0.818182,0.128571
24,8,0.99,0.818182,0.128571
27,10,0.90,0.818182,0.128571
28,10,0.95,0.818182,0.128571
29,10,0.99,0.818182,0.128571
20,8,0.70,0.733333,0.157143
21,8,0.80,0.733333,0.157143
25,10,0.70,0.733333,0.157143
26,10,0.80,0.733333,0.157143


In [ ]:
MIN_PAL_IDIOMA, UMBRAL_IDIOMA = 8, 0.90

info_test = prueba["synopsis"].apply(info_oraciones)
y_pred_idioma = info_test.apply(lambda i: marcar(i, MIN_PAL_IDIOMA, UMBRAL_IDIOMA))
medir("idioma por oración", y_pred_idioma, prueba)

idioma por oración: precisión 0.64, recall 0.30


(0.6428571428571429, 0.3)

#### Justificación de los valores

En la tabla de desarrollo, con oraciones de menos de 8 palabras la precisión no supera 0.70: las oraciones cortas, las listas de nombres propios y las citas se confunden con otros idiomas. Con **8 palabras** la precisión sube a 0.82 y ya no mejora con 10. A partir de un umbral de **0.90** los resultados no cambian, porque langdetect suele estar muy seguro cuando se equivoca. Se elige `MIN_PAL_IDIOMA = 8` y `UMBRAL_IDIOMA = 0.90`.

Aun así el detector es débil: en prueba tiene precisión ≈0.64 y recall ≈0.30. Sus falsos positivos son oraciones en castellano con muchos nombres extranjeros, títulos en otro idioma o texto todo en mayúsculas.

## 2.4

In [ ]:
from gliner import GLiNER
from collections import Counter
import random

modelo_gliner = GLiNER.from_pretrained("urchade/gliner_multi_pii-v1")
etiquetas = ["persona", "organización", "lugar", "sitio web", "marca comercial", "precio", "correo electrónico"]

# Oraciones de los fragmentos contaminados y la misma cantidad de oraciones limpias
frag = desarrollo[desarrollo["contaminado"] == 1]["fragmento"]
oraciones_contaminadas = [o for f in frag for o in (oraciones(f) or [f])]
pool = [o for s in desarrollo[desarrollo["contaminado"] == 0]["synopsis"] for o in oraciones(s)]
oraciones_limpias = random.Random(42).sample(pool, len(oraciones_contaminadas))

def extraer_entidades(textos, modelo, etiquetas):
    resultados = []
    for texto in textos:
        for ent in modelo.predict_entities(texto, etiquetas, threshold=0.5):
            resultados.append((ent["text"], ent["label"]))
    return resultados

entidades_contaminadas = extraer_entidades(oraciones_contaminadas, modelo_gliner, etiquetas)
entidades_limpias = extraer_entidades(oraciones_limpias, modelo_gliner, etiquetas)

print("Etiquetas en CONTAMINADOS")
print(Counter(e for _, e in entidades_contaminadas))
print("\nEtiquetas en LIMPIOS")
print(Counter(e for _, e in entidades_limpias))

print("\nEjemplos CONTAMINADOS")
for texto, etiqueta in list(set(entidades_contaminadas))[:15]:
    print(f"[{etiqueta.upper()}] {texto}")

Etiquetas en CONTAMINADOS
Counter({'sitio web': 16, 'organización': 16, 'lugar': 5})

Etiquetas en LIMPIOS
Counter({'persona': 49, 'lugar': 25, 'organización': 18, 'sitio web': 1, 'marca comercial': 1})

Ejemplos CONTAMINADOS
[SITIO WEB] tohub.test
[ORGANIZACIÓN] LetraTotal
[ORGANIZACIÓN] w w w .
[ORGANIZACIÓN] CapituloPlus
[SITIO WEB] lectorya.test
[SITIO WEB] https://capituloclub.example/no-las-llames-chicas-llamalas-
[SITIO WEB] http://www.letramax.invalid
[ORGANIZACIÓN] LectorPlus
[SITIO WEB] http://www.bibliozona.example
[SITIO WEB] librohub.test
[ORGANIZACIÓN] TomoHub
[ORGANIZACIÓN] LetraMax
[ORGANIZACIÓN] escuela-de-ajedrez
[SITIO WEB] http://www.lectorexpress.test
[ORGANIZACIÓN] LibroLibre


### Análisis de las entidades detectadas

En las oraciones contaminadas, GLiNER detectó principalmente entidades de tipo sitio web (16 apariciones) y, en menor medida, organización (16) y lugar (5). En cambio, las oraciones limpias estuvieron dominadas por entidades de tipo persona (49), seguidas por lugar (25) y organización (18), mientras que sitio web apareció solo una vez y marca comercial una vez.

Estos resultados muestran que las entidades sitio web son un fuerte indicador de contaminación, ya que prácticamente no aparecen en las sinopsis legítimas y suelen corresponder a enlaces o páginas utilizadas para promocionar la descarga de libros. Por el contrario, las entidades persona, lugar y organización también son frecuentes en las sinopsis normales, ya que forman parte del contenido propio de la descripción de un libro.

En consecuencia, GLiNER resulta útil principalmente como una herramienta de análisis e interpretación de la contaminación. Permite explicar por qué determinadas sinopsis fueron marcadas (por ejemplo, por la presencia de sitios web o marcas comerciales), pero por sí solo no constituye un detector suficientemente discriminativo, ya que la mayoría de las entidades identificadas también aparecen en textos legítimos.

## 2.5 Clasificar oraciones

#### Construcción del conjunto de oraciones

Las oraciones del `fragmento` insertado son positivas y las oraciones de las sinopsis limpias son negativas. Se usa la misma función `oraciones()` de la guía para entrenar y para aplicar el detector.

In [34]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_curve

def armar_oraciones(libros):
    filas = []
    for _, libro in libros.iterrows():
        if libro["contaminado"] == 1:
            for o in oraciones(libro["fragmento"]) or [libro["fragmento"]]:
                filas.append({"texto": o, "contaminado": 1})
        else:
            for o in oraciones(libro["synopsis"]):
                filas.append({"texto": o, "contaminado": 0})
    return pd.DataFrame(filas)

oraciones_df = armar_oraciones(desarrollo)
print(oraciones_df["contaminado"].value_counts())
display(oraciones_df[oraciones_df["contaminado"] == 1].head())

contaminado
0    2007
1      88
Name: count, dtype: int64


,texto,contaminado
56,El libro más esperado del año llega por fin en...,1
57,Leé este y miles de libros más sin publicidad.,1
58,Activá tu prueba gratuita desde tu celular.,1
71,"Download this ebook for FREE, no sign-up requi...",1
85,Una historia apasionante que te atrapará desde...,1


#### Los dos clasificadores

Los dos usan `LogisticRegression(class_weight="balanced", max_iter=1000)`. El primero representa cada oración con TF-IDF y el segundo con el modelo de embeddings multilingüe del TP2 (`distiluse-base-multilingual-cased-v1`). Una sinopsis se puntúa con la probabilidad más alta entre sus oraciones.

In [ ]:
from sentence_transformers import SentenceTransformer

modelo_emb = SentenceTransformer("distiluse-base-multilingual-cased-v1")

_embeddings = {}
def codificar(textos):
    textos = list(textos)
    nuevos = [t for t in dict.fromkeys(textos) if t not in _embeddings]
    if nuevos:
        for t, v in zip(nuevos, modelo_emb.encode(nuevos, batch_size=64)):
            _embeddings[t] = v
    return np.vstack([_embeddings[t] for t in textos])

def entrenar_tfidf(textos, y):
    modelo = Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000))
    ])
    modelo.fit(list(textos), y)
    return lambda nuevos: modelo.predict_proba(list(nuevos))[:, 1]

def entrenar_embeddings(textos, y):
    clf = LogisticRegression(class_weight="balanced", max_iter=1000)
    clf.fit(codificar(textos), y)
    return lambda nuevos: clf.predict_proba(codificar(nuevos))[:, 1]

def puntaje_libros(proba, sinopsis):
    """Probabilidad más alta entre las oraciones de cada sinopsis."""
    por_libro = [oraciones(s) or [s] for s in sinopsis]
    p = proba([o for ors in por_libro for o in ors])
    puntajes, k = [], 0
    for ors in por_libro:
        puntajes.append(p[k:k + len(ors)].max())
        k += len(ors)
    return np.array(puntajes)

#### Selección del umbral

No se usa 0.5: los dos modelos no producen probabilidades comparables. El umbral se elige sobre la muestra de desarrollo con `precision_recall_curve`, **a nivel de libro**: con validación cruzada de 5 particiones por libro, cada sinopsis de desarrollo se puntúa con un modelo que no vio sus oraciones (las mismas plantillas promocionales se repiten en varios libros, así que separar por oración daría un resultado optimista). Se toma el menor umbral que alcanza una precisión de 0.90.

In [36]:
def elegir_umbral(entrenar, precision_objetivo=0.90):
    puntajes = np.zeros(len(desarrollo))
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for idx_tr, idx_va in folds.split(desarrollo, desarrollo["contaminado"]):
        ors = armar_oraciones(desarrollo.iloc[idx_tr])
        proba = entrenar(ors["texto"], ors["contaminado"])
        puntajes[idx_va] = puntaje_libros(proba, desarrollo["synopsis"].iloc[idx_va])
    precision, recall, umbrales = precision_recall_curve(desarrollo["contaminado"], puntajes)
    curva = pd.DataFrame({"umbral": umbrales, "precision": precision[:-1], "recall": recall[:-1]})
    return curva[curva["precision"] >= precision_objetivo].iloc[0], curva

fila_tfidf, curva_tfidf = elegir_umbral(entrenar_tfidf)
fila_emb, curva_emb = elegir_umbral(entrenar_embeddings)
UMBRAL_PROMO_TFIDF, UMBRAL_PROMO_EMB = fila_tfidf["umbral"], fila_emb["umbral"]

display(pd.DataFrame([fila_tfidf, fila_emb], index=["TF-IDF", "embeddings"]).round(3))

,umbral,precision,recall
TF-IDF,0.441,0.901,0.914
embeddings,0.567,0.903,0.929


#### Evaluación sobre el conjunto de prueba

Con el umbral elegido, cada modelo se entrena con todas las oraciones de desarrollo y se aplica a las sinopsis de prueba.

In [37]:
proba_tfidf = entrenar_tfidf(oraciones_df["texto"], oraciones_df["contaminado"])
proba_emb = entrenar_embeddings(oraciones_df["texto"], oraciones_df["contaminado"])

def detectar_promocion_tfidf(sinopsis):
    return (puntaje_libros(proba_tfidf, sinopsis) >= UMBRAL_PROMO_TFIDF).astype(int)

def detectar_promocion_emb(sinopsis):
    return (puntaje_libros(proba_emb, sinopsis) >= UMBRAL_PROMO_EMB).astype(int)

y_pred_tfidf = pd.Series(detectar_promocion_tfidf(prueba["synopsis"]), index=prueba.index)
y_pred_emb = pd.Series(detectar_promocion_emb(prueba["synopsis"]), index=prueba.index)

medir("clasificador TF-IDF", y_pred_tfidf, prueba)
medir("clasificador embeddings", y_pred_emb, prueba)

clasificador TF-IDF: precisión 1.00, recall 0.93
clasificador embeddings: precisión 0.90, recall 0.87


(0.896551724137931, 0.8666666666666667)

In [ ]:
# Contaminados de prueba que detecta cada uno
comparacion = prueba.loc[prueba["contaminado"] == 1, ["fragmento"]].copy()
comparacion["TF-IDF"] = y_pred_tfidf
comparacion["embeddings"] = y_pred_emb
comparacion["fragmento"] = comparacion["fragmento"].str[:90]
display(comparacion[comparacion["TF-IDF"] != comparacion["embeddings"]])
print("Ninguno de los dos detecta:")
for f in comparacion[(comparacion["TF-IDF"] == 0) & (comparacion["embeddings"] == 0)]["fragmento"]:
    print("-", f)

,fragmento,TF-IDF,embeddings
285,Entrá a letrаya.invalid y leé sin límites.,1,0
27,Descargalo gratis en TomoMax punto example bar...,1,0


Ninguno de los dos detecta:
- Pedí tu copia escribiendo a ventas@tomozona.test
- Link en nuestro sitio: h t t p s : / / lectortotal [.] example


#### Respuestas 2.5

**¿Por qué clasificar oraciones y no la sinopsis completa?** El texto insertado es una o dos oraciones dentro de una sinopsis legítima. Si se clasifica la sinopsis completa, la señal del mensaje promocional se diluye entre el resto del texto (lo mismo que pasó con el idioma en 2.3). Con oraciones, el modelo aprende exactamente cómo es el texto insertado y la sinopsis se marca si **alguna** oración lo parece.

**¿Qué representación detecta mejor los contaminados de prueba?** A igual precisión en desarrollo (0.90), TF-IDF obtiene en prueba precisión ≈1.00 y recall ≈0.93; embeddings, precisión ≈0.90 y recall ≈0.87. TF-IDF gana en esta muestra.

**¿En qué tipo de contaminación gana cada una?** TF-IDF detecta mejor las **cadenas literales**: palabras como «descargá», «gratis», «PDF», «EPUB», nombres de dominio o de las apps (LetraClub, EpubMax), que se repiten idénticas entre libros. Los embeddings representan el **sentido** de la oración, así que pierden los enlaces disimulados (un dominio raro no tiene un significado parecido a nada) pero son los candidatos a generalizar mejor a mensajes nuevos con el mismo sentido y otras palabras u otro idioma. Como en el catálogo hay mensajes que no aparecen en la muestra, esa ventaja de los embeddings puede ser importante. Sin embargo, en esta muestra los mensajes promocionales son plantillas que se repiten, por eso TF-IDF rinde mejor. Las URL las detecta mejor TF-IDF, y aun mejor la regex.

## 2.6

#### Construcción de la representación TF-IDF y búsqueda de vecinos más cercanos

In [39]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

vectorizador = TfidfVectorizer(
    lowercase=True,
    stop_words=None,
    min_df=2
)

X = vectorizador.fit_transform(df["synopsis"])

nn = NearestNeighbors(
    n_neighbors=2,
    metric="cosine"
)

nn.fit(X)

distancias, vecinos = nn.kneighbors(X)

#### Obtención del vecino más similar para cada libro

In [40]:
pares = []

for i in range(len(df)):

    vecino = vecinos[i,1]

    similitud = 1 - distancias[i,1]

    pares.append({
        "idx1": i,
        "idx2": vecino,
        "autor1": df.iloc[i]["authors"],
        "autor2": df.iloc[vecino]["authors"],
        "titulo1": df.iloc[i]["title"],
        "titulo2": df.iloc[vecino]["title"],
        "similitud": similitud
    })

pares = pd.DataFrame(pares)

#### Filtrado de candidatos con autores distintos y alta similitud

In [41]:
pares = pares[
    pares["autor1"] != pares["autor2"]
]

pares = pares.sort_values(
    "similitud",
    ascending=False
)

display(pares.head(20))

,idx1,idx2,autor1,autor2,titulo1,titulo2,similitud
4201,4201,1334,[Gemma Elwin Harris],[Varios Autores],Las grandes preguntas de los niños y las senci...,Las grandes preguntas de los niños y las senci...,1.000000
8342,8342,5345,[Paula Arrieta],[Juan Villalba Sebastián],Los hijos de la promesa,Elvira de Hidalgo,0.988401
5345,5345,8342,[Juan Villalba Sebastián],[Paula Arrieta],Elvira de Hidalgo,Los hijos de la promesa,0.988401
5036,5036,2083,[Lucía Oyarzún],[Lola Lafon],La última tormenta,La pequeña comunista que no sonreía nunca,0.988012
2083,2083,5036,[Lola Lafon],[Lucía Oyarzún],La pequeña comunista que no sonreía nunca,La última tormenta,0.988012
7957,7957,6764,[Julio Llamazares],[Clara Correa],Trás-os-Montes,Bajo la isla,0.984944
6764,6764,7957,[Clara Correa],[Julio Llamazares],Bajo la isla,Trás-os-Montes,0.984944
270,270,3414,[Richard Phillips Feynman],[Michelle Feynman],La física de las palabras,Richard P. Feynman. La física de las palabras,0.983809
3414,3414,270,[Michelle Feynman],[Richard Phillips Feynman],Richard P. Feynman. La física de las palabras,La física de las palabras,0.983809
7125,7125,5751,[Bjørn Lomborg],[Andrés Barrionuevo],En frío,La última herencia,0.977142


#### Implementación de la similitud de Jaccard para validar los candidatos

In [42]:
import re

def jaccard(a, b):

    A = set(re.findall(r"\w+", a.lower()))
    B = set(re.findall(r"\w+", b.lower()))

    if len(A | B) == 0:
        return 0

    return len(A & B) / len(A | B)

#### Comparación de la similitud del coseno y la similitud de Jaccard

In [43]:
pares["jaccard"] = pares.apply(

    lambda fila: jaccard(df.iloc[fila["idx1"]]["synopsis"], df.iloc[fila["idx2"]]["synopsis"]),axis=1)

display(
    pares.sort_values(
        ["similitud","jaccard"],
        ascending=False
    ).head(20)
)

,idx1,idx2,autor1,autor2,titulo1,titulo2,similitud,jaccard
4201,4201,1334,[Gemma Elwin Harris],[Varios Autores],Las grandes preguntas de los niños y las senci...,Las grandes preguntas de los niños y las senci...,1.000000,1.000000
8342,8342,5345,[Paula Arrieta],[Juan Villalba Sebastián],Los hijos de la promesa,Elvira de Hidalgo,0.988401,0.954198
5345,5345,8342,[Juan Villalba Sebastián],[Paula Arrieta],Elvira de Hidalgo,Los hijos de la promesa,0.988401,0.954198
5036,5036,2083,[Lucía Oyarzún],[Lola Lafon],La última tormenta,La pequeña comunista que no sonreía nunca,0.988012,0.944251
2083,2083,5036,[Lola Lafon],[Lucía Oyarzún],La pequeña comunista que no sonreía nunca,La última tormenta,0.988012,0.944251
7957,7957,6764,[Julio Llamazares],[Clara Correa],Trás-os-Montes,Bajo la isla,0.984944,0.954128
6764,6764,7957,[Clara Correa],[Julio Llamazares],Bajo la isla,Trás-os-Montes,0.984944,0.954128
270,270,3414,[Richard Phillips Feynman],[Michelle Feynman],La física de las palabras,Richard P. Feynman. La física de las palabras,0.983809,0.925620
3414,3414,270,[Michelle Feynman],[Richard Phillips Feynman],Richard P. Feynman. La física de las palabras,La física de las palabras,0.983809,0.925620
7125,7125,5751,[Bjørn Lomborg],[Andrés Barrionuevo],En frío,La última herencia,0.977142,0.936416


#### Inspección manual de los pares más similares

In [44]:
for _, fila in pares.head(10).iterrows():

    print("=" * 100)

    print(f"Similitud coseno: {fila['similitud']:.3f}")
    print(f"Jaccard: {fila['jaccard']:.3f}\n")

    print("LIBRO 1")
    print(f"Título: {fila['titulo1']}")
    print(f"Autor : {fila['autor1']}")
    print("Sinopsis:")
    print(df.iloc[fila["idx1"]]["synopsis"])
    print("")
    print("LIBRO 2")
    print(f"Título: {fila['titulo2']}")
    print(f"Autor : {fila['autor2']}")
    print("Sinopsis:")
    print(df.iloc[fila["idx2"]]["synopsis"])

Similitud coseno: 1.000
Jaccard: 1.000

LIBRO 1
Título: Las grandes preguntas de los niños y las sencillas respuestas de los grandes expertos
Autor : ['Gemma Elwin Harris']
Sinopsis:
«¿Por qué el espacio es tan brillante?», «¿Quién tuvo la primera mascota?». «¿Puede una abeja picar a otra?». «¿Cómo se hace la electricidad?». «¿De dónde vienen los océanos?». «¿Por qué hay guerras?», «¿Cómo se enamora la gente?». «¿De dónde viene el bien?». Todas estas preguntas y muchas más, realizadas por niños de todas las edades, inspiran este maravilloso libro en el que han participado los mayores expertos del mundo para responder a la curiosidad de los más pequeños de la casa y también a la de los adultos. De forma entretenida y divertida, la escritora Gemma Elwin Harris ha recopilado grandes preguntas de alumnos de educación primaria y ha reunido a un extraordinario equipo de científicos, especialistas, filósofos y escritores para que las respondan. Expertos como Richard Dawkins, Noam Chomsky, Sir

### Análisis de los pares con mayor similitud

Se revisaron manualmente los pares con mayor similitud del coseno y autores distintos para determinar si correspondían a una copia de la sinopsis de otro libro o a un caso legítimo (reediciones o diferencias en la autoría registrada).

- Las grandes preguntas de los niños y las sencillas respuestas de los grandes expertos
  - Original: Registro con autora Gemma Elwin Harris.
  - Registro duplicado: Registro con autor "Varios Autores".
  - Justificación: Las sinopsis son prácticamente idénticas (similitud del coseno y Jaccard iguales a 1). No se observa texto agregado, por lo que se trata de un registro duplicado o una diferencia en la catalogación del autor, más que de una copia con modificaciones.

- Elvira de Hidalgo - Los hijos de la promesa
  - Original: *Elvira de Hidalgo* (Juan Villalba Sebastián).
  - Copia: *Los hijos de la promesa* (Paula Arrieta).
  - Justificación: La sinopsis de *Los hijos de la promesa* comienza con un breve texto promocional ("Un relato inolvidable..." y un sitio de descarga) y luego copia íntegramente la sinopsis de *Elvira de Hidalgo*. Es un caso claro de reutilización de una sinopsis con agregado de contenido promocional.

- Bajo la isla - Trás-os-Montes
  - Original: *Trás-os-Montes* (Julio Llamazares).
  - Copia: *Bajo la isla* (Clara Correa).
  - Justificación: La sinopsis corresponde completamente al libro *Trás-os-Montes* y fue reutilizada para un libro con otro título y otro autor, lo que indica un caso de copia.

- La física de las palabras - Richard P. Feynman. La física de las palabras
  - Original: No puede determinarse.
  - Justificación: Ambas sinopsis describen exactamente el mismo libro y solo difieren en una frase introductoria y en la forma de registrar el autor y el título. Todo indica que corresponde a una reedición o una diferencia editorial, no a una copia fraudulenta.

- La última herencia - En frío
  - Original: *En frío* (Bjørn Lomborg).
  - Copia: *La última herencia* (Andrés Barrionuevo).
  - Justificación: La sinopsis de *La última herencia* agrega un encabezado promocional y datos de contacto, pero el resto del texto coincide con la sinopsis de *En frío*. Se trata de un caso claro de copia con contenido promocional añadido.

En conjunto, la búsqueda de vecinos más cercanos permitió detectar tanto registros duplicados como casos en los que una sinopsis fue reutilizada para un libro diferente. Los ejemplos más claros de copia son aquellos en los que la sinopsis del libro sospechoso contiene íntegramente la del libro original y agrega únicamente frases promocionales o información de contacto al comienzo.

#### Detector de copias

La revisión manual muestra un patrón claro: la copia contiene **todas** las palabras de la sinopsis original y le agrega algunas (un encabezado promocional o datos de contacto), mientras que el original no agrega nada respecto de la copia. Por eso, para cada libro se mide qué parte de las palabras de su vecino está contenida en su propia sinopsis (Jaccard asimétrico) y cuántas palabras nuevas agrega.

Un libro se marca como copia si:
- la similitud del coseno con su vecino es alta (≥ 0.8),
- contiene al menos el 90 % de las palabras del vecino y agrega 3 o más,
- y los autores son distintos: no comparten apellido (así se descartan casos como *La física de las palabras*, de Richard y Michelle Feynman) ni pertenecen a la misma serie.

In [45]:
def palabras(texto):
    return set(re.findall(r"\w+", texto.lower()))

def apellidos(autores):
    return {p for a in autores for p in a.lower().split() if len(p) > 3}

copias = []
for i in range(len(df)):
    pos = 1 if vecinos[i, 0] == i else 0
    j = vecinos[i, pos]
    similitud = 1 - distancias[i, pos]
    A, B = palabras(df["synopsis"].iloc[i]), palabras(df["synopsis"].iloc[j])
    contiene = len(A & B) / len(B)
    agregadas = len(A - B)
    mismo_autor = bool(apellidos(df["authors"].iloc[i]) & apellidos(df["authors"].iloc[j]))
    misma_serie = isinstance(df["series"].iloc[i], str) and df["series"].iloc[i] == df["series"].iloc[j]

    if similitud >= 0.8 and contiene >= 0.9 and agregadas >= 3 and not mismo_autor and not misma_serie:
        copias.append({
            "slug": df["slug"].iloc[i],
            "copia": df["title"].iloc[i],
            "autor copia": ", ".join(df["authors"].iloc[i]),
            "original": df["title"].iloc[j],
            "autor original": ", ".join(df["authors"].iloc[j]),
            "similitud": round(similitud, 3),
            "contiene": round(contiene, 3),
            "agregadas": agregadas,
        })

copias = pd.DataFrame(copias)
print(f"Libros marcados como copia en el catálogo: {len(copias)}")
display(copias.head(10))

Libros marcados como copia en el catálogo: 62


,slug,copia,autor copia,original,autor original,similitud,contiene,agregadas
0,la-ultima-estacion,La última estación,Ignacio Barrionuevo,Patty Diphusa,Pedro Almodóvar,0.956,1.0,16
1,cronica-del-faro-4,Crónica del faro,Tomás Vidal,Unidos por el escándalo,Eleanor Rigby,0.942,1.0,18
2,bajo-la-tormenta,Bajo la tormenta,Lucía Vidal,Viaje a la revolución,Bertrand Russell,0.949,1.0,13
3,bajo-la-isla-2,Bajo la isla,Martín Arrieta,La condición obrera,Simone Weil,0.958,1.0,19
4,el-secreto-del-faro,El secreto del faro,Valeria Peralta,ISIS,Patrick Cockburn,0.960,1.0,19
5,el-camino-de-la-llanura-2,El camino de la llanura,Valeria Saravia,La Edad del Futuro I,Isaac Asimov,0.935,1.0,17
6,cronica-del-silencio-2,Crónica del silencio,Martín Barrionuevo,La vida cotidiana en Pompeya,Robert Étienne,0.942,1.0,16
7,la-ultima-colmena,La última colmena,Tomás Correa,Hamlet (Ed. Clásicos a medida),William Shakespeare,0.921,1.0,17
8,el-camino-de-la-estacion-2,El camino de la estación,Lucía Peralta,El peregrino de Compostela,Paulo Coelho,0.935,1.0,14
9,sombras-sobre-el-bosque,Sombras sobre el bosque,Paula Oyarzún,Bodas de sangre,Federico García Lorca,0.892,1.0,21


In [ ]:
y_pred_copia = prueba["slug"].isin(set(copias["slug"])).astype(int)
medir("copias (vecino más parecido)", y_pred_copia, prueba)

copias (vecino más parecido): precisión 1.00, recall 0.10


(1.0, 0.1)

#### Respuesta 2.6

El detector marca 62 libros del catálogo y en prueba tiene precisión 1.00. El recall es bajo (≈0.10) porque detecta un solo tipo de contaminación: los libros inventados. En la mayoría de las copias el texto agregado es promocional, así que la regex o el clasificador también las marcan. Aun así, este detector es el único que explica por qué el libro es falso (la sinopsis es de otro libro) y encuentra copias aunque el texto agregado no tenga nada promocional.

Los libros marcados tienen títulos genéricos («Memorias del faro», «Los hijos de la llanura», «Crónica del bosque») y autores que repiten unos pocos apellidos (Correa, Peralta, Vidal, Barrionuevo, Oyarzún...), lo que confirma que son copias. En cambio, quedan sin marcar los pares legítimos que comparten texto: los tomos de una serie (mismo autor o misma serie) y los dos registros de *Las grandes preguntas de los niños...*, cuyas sinopsis son idénticas y no agregan texto (son un duplicado de catalogación, no una copia).

## 2.7 Combinación de detectores

Cada detector encuentra un tipo distinto de contaminación. Se combinan con una regla OR: una sinopsis es sospechosa si al menos uno de los detectores elegidos la marca. Primero se comparan todos sobre la parte de prueba de la muestra.

In [47]:
detectores_prueba = {
    "regex": y_pred_regex,
    "normalizacion": y_pred_norm,
    "idioma": y_pred_idioma,
    "clasificador TF-IDF": y_pred_tfidf,
    "clasificador embeddings": y_pred_emb,
    "casi_duplicado": y_pred_copia,
}

def fila_metricas(marcado):
    y = prueba["contaminado"]
    return {
        "precisión": round(precision_score(y, marcado, zero_division=0), 2),
        "recall": round(recall_score(y, marcado), 2),
        "marcados": int(marcado.sum()),
        "falsos positivos": int(((marcado == 1) & (y == 0)).sum()),
    }

def combinar(nombres):
    return pd.concat([detectores_prueba[n] for n in nombres], axis=1).max(axis=1)

COMBINACION = ["regex", "normalizacion", "clasificador TF-IDF", "casi_duplicado"]

tabla = {n: fila_metricas(m) for n, m in detectores_prueba.items()}
tabla["OR de todos"] = fila_metricas(combinar(detectores_prueba))
tabla["OR elegido: " + " + ".join(COMBINACION)] = fila_metricas(combinar(COMBINACION))
display(pd.DataFrame(tabla).T)

,precisión,recall,marcados,falsos positivos
regex,1.00,0.40,12.0,0.0
normalizacion,1.00,0.63,19.0,0.0
idioma,0.64,0.30,14.0,5.0
clasificador TF-IDF,1.00,0.93,28.0,0.0
clasificador embeddings,0.90,0.87,29.0,3.0
casi_duplicado,1.00,0.10,3.0,0.0
OR de todos,0.83,1.00,36.0,6.0
OR elegido: regex + normalizacion + clasificador TF-IDF + casi_duplicado,1.00,1.00,30.0,0.0


#### Combinación elegida

La combinación final usa **regex + normalización + clasificador TF-IDF + copias**. En prueba alcanza recall 1.00 sin falsos positivos.

- **Idioma** no se incluye: no agrega ningún libro contaminado que los otros no encuentren, pero sí falsos positivos. En el catálogo marcaría unos 140 libros más, en su mayoría sinopsis legítimas con nombres o citas en otro idioma.
- **Clasificador con embeddings** tampoco se incluye: tiene menos precisión que el de TF-IDF y no suma recall.

Marcar todo no sirve: cada libro legítimo que se elimina es un libro que nunca se va a poder recomendar.

Hay que tener en cuenta que la muestra está enriquecida (20 % de contaminados) y que en el catálogo la proporción es menor, así que la precisión real será más baja que en prueba. Al revisar los libros que marca solo el clasificador TF-IDF, la mayoría tiene mensajes promocionales reales sin enlace («Ganá dinero desde tu casa...», «Leia milhares de livros...»), pero también hay sinopsis legítimas escritas en segunda persona («¿Te atreves a arriesgar tu corazón?»). Ese es el costo de este detector.

#### Aplicación al catálogo completo

In [48]:
info_catalogo = df["synopsis"].apply(info_oraciones)

detectores_catalogo = {
    "regex": df["synopsis"].apply(predecir_con_regex),
    "normalizacion": df["synopsis"].apply(predecir_normalizado),
    "idioma": info_catalogo.apply(lambda i: marcar(i, MIN_PAL_IDIOMA, UMBRAL_IDIOMA)),
    "clasificador TF-IDF": pd.Series(detectar_promocion_tfidf(df["synopsis"]), index=df.index),
    "casi_duplicado": df["slug"].isin(set(copias["slug"])).astype(int),
}

display(pd.Series({n: int(m.sum()) for n, m in detectores_catalogo.items()}, name="libros marcados en el catálogo"))

regex                  131
normalizacion          203
idioma                 176
clasificador TF-IDF    306
casi_duplicado          62
Name: libros marcados en el catálogo, dtype: int64

In [49]:
MOTIVO = {
    "regex": "regex",
    "normalizacion": "normalizacion",
    "idioma": "idioma",
    "clasificador TF-IDF": "clasificador",
    "clasificador embeddings": "clasificador",
    "casi_duplicado": "casi_duplicado",
}

def motivo_libro(i):
    motivos = []
    for n in COMBINACION:
        if detectores_catalogo[n].iloc[i] and MOTIVO[n] not in motivos:
            motivos.append(MOTIVO[n])
    return "+".join(motivos)

df["motivo"] = [motivo_libro(i) for i in range(len(df))]
sospechosos = df.loc[df["motivo"] != "", ["slug", "motivo"]]

print(f"Libros sospechosos: {len(sospechosos)} de {len(df)} ({len(sospechosos) / len(df):.1%})")
display(sospechosos["motivo"].value_counts())

Libros sospechosos: 352 de 8640 (4.1%)


motivo
clasificador                                       130
regex+normalizacion+clasificador                    72
normalizacion+clasificador                          48
regex+normalizacion                                 30
regex+normalizacion+clasificador+casi_duplicado     26
clasificador+casi_duplicado                         18
normalizacion+clasificador+casi_duplicado           12
normalizacion                                       10
regex+normalizacion+casi_duplicado                   3
normalizacion+casi_duplicado                         2
casi_duplicado                                       1
Name: count, dtype: int64

In [50]:
GRUPO = "grupo3"

sospechosos.to_csv(f"{GRUPO}_sospechosos.csv", index=False)

catalogo_limpio = df[df["motivo"] == ""].drop(columns="motivo")
catalogo_limpio.to_csv("catalogo_limpio.csv", index=False)

print(f"Libros sospechosos : {len(sospechosos)}")
print(f"Catálogo limpio    : {len(catalogo_limpio)}")

Libros sospechosos : 352
Catálogo limpio    : 8288


# PASO 3
## Registrar las preferencias de un lector

Lector de referencia: **Manuel Lazarte**, quien realizó todas las valoraciones.

Se mezcla el catálogo limpio una sola vez con `random_state=42` y el lector recorre ese orden desde el principio, respondiendo para cada libro «¿Me interesaría leer este libro?» con 1 (sí), 0 (no) o dejando vacío si no puede decidir.

In [ ]:
catalogo_limpio = pd.read_csv("catalogo_limpio.csv")

orden = catalogo_limpio.sample(frac=1, random_state=42).copy()
orden_2 = catalogo_limpio.sample(frac=1, random_state=42).copy()

print(orden["slug"].equals(orden_2["slug"]))

True


#### Planilla para valorar

Se exportan los primeros 200 libros del orden en `planilla_valoracion.csv`. El lector completa la columna `interes` en el mismo orden (1, 0 o vacío) hasta reunir al menos 100 respuestas válidas y 20 de cada clase, y la guarda como `preferencias.csv`. Si encuentra un libro contaminado que el paso 2 no detectó, lo deja vacío y anota su slug en `revision_manual`.

In [52]:
import os

if not os.path.exists("preferencias.csv"):
    planilla = orden[["slug", "title", "synopsis"]].head(200).copy()
    planilla["interes"] = ""
    planilla.to_csv("planilla_valoracion.csv", index=False)
    print("Se generó planilla_valoracion.csv: completarla y guardarla como preferencias.csv")

In [ ]:
preferencias = pd.read_csv("preferencias.csv")

# Se conservan los libros hasta el último que el lector vio y valoró
# Los vacíos intermedios son abstenciones y quedan con interes vacío (NaN), no se convierten en 0.
ultimo_valorado = preferencias["interes"].last_valid_index()
preferencias = preferencias.loc[:ultimo_valorado, ["slug", "title", "synopsis", "interes"]]

print(f"Libros vistos: {len(preferencias)}")
print(preferencias["interes"].value_counts(dropna=False).rename({1.0: "1 (sí)", 0.0: "0 (no)"}))

conteo = preferencias["interes"].value_counts()
suficientes = conteo.sum() >= 100 and conteo.get(1, 0) >= 20 and conteo.get(0, 0) >= 20
print(f"\n¿Al menos 100 respuestas válidas y 20 de cada clase? {suficientes}")

Libros vistos: 200
interes
0 (no)    140
NaN        32
1 (sí)     28
Name: count, dtype: int64

¿Al menos 100 respuestas válidas y 20 de cada clase? True


In [ ]:
# Libros contaminados que encontró el lector y que el paso 2 no había detectado
revision_manual = []   

if revision_manual:
    sospechosos = pd.concat([
        sospechosos,
        pd.DataFrame({"slug": revision_manual, "motivo": "revision_manual"})
    ]).drop_duplicates(subset="slug")
    sospechosos.to_csv(f"{GRUPO}_sospechosos.csv", index=False)

print(f"Libros marcados en revisión manual: {len(revision_manual)}")

Libros marcados en revisión manual: 0


# PASO 4
## Separar los libros para aprender y evaluar

In [55]:
validas = preferencias.dropna(subset=["interes"]).copy()
validas["interes"] = validas["interes"].astype(int)

pref_entrenamiento, pref_prueba = train_test_split(
    validas,
    test_size=0.2,
    random_state=42,
    stratify=validas["interes"]
)

preferencias["conjunto"] = ""
preferencias.loc[pref_entrenamiento.index, "conjunto"] = "entrenamiento"
preferencias.loc[pref_prueba.index, "conjunto"] = "prueba"
preferencias.to_csv(f"{GRUPO}_preferencias.csv", index=False)

display(pd.DataFrame({
    "libros": [len(pref_entrenamiento), len(pref_prueba)],
    "interes = 1": [pref_entrenamiento["interes"].sum(), pref_prueba["interes"].sum()]
}, index=["entrenamiento", "prueba"]))

,libros,interes = 1
entrenamiento,134,22
prueba,34,6


# PASO 5
## Entrenar dos clasificadores

Ambos modelos usan la misma configuración fija: `LogisticRegression(C=1, max_iter=1000, class_weight="balanced")`. `class_weight="balanced"` le da más peso a los errores en la clase menos frecuente; sin esto, con pocos positivos el modelo tiende a predecir siempre la clase mayoritaria.

#### Modelo A con TF-IDF

In [56]:
y_pref_train = pref_entrenamiento["interes"]
y_pref_test = pref_prueba["interes"]

vectorizador_pref = TfidfVectorizer()
X_pref_train_tfidf = vectorizador_pref.fit_transform(pref_entrenamiento["synopsis"])
X_pref_test_tfidf = vectorizador_pref.transform(pref_prueba["synopsis"])

modelo_a = LogisticRegression(C=1, max_iter=1000, class_weight="balanced")
modelo_a.fit(X_pref_train_tfidf, y_pref_train)
pred_a = modelo_a.predict(X_pref_test_tfidf)

print("Predicciones modelo A:", pred_a.tolist())

Predicciones modelo A: [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


#### Modelo B con embeddings

Se reutiliza el modelo multilingüe del TP2 (`distiluse-base-multilingual-cased-v1`), ya cargado en el paso 2.5. El modelo de embeddings queda fijo; solo se entrena la regresión logística.

In [57]:
X_pref_train_emb = modelo_emb.encode(pref_entrenamiento["synopsis"].tolist())
X_pref_test_emb = modelo_emb.encode(pref_prueba["synopsis"].tolist())

modelo_b = LogisticRegression(C=1, max_iter=1000, class_weight="balanced")
modelo_b.fit(X_pref_train_emb, y_pref_train)
pred_b = modelo_b.predict(X_pref_test_emb)

print("Predicciones modelo B:", pred_b.tolist())

Predicciones modelo B: [1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0]


In [58]:
print(f"TF-IDF     -> entrenamiento {X_pref_train_tfidf.shape}, prueba {X_pref_test_tfidf.shape}")
print(f"Embeddings -> entrenamiento {X_pref_train_emb.shape}, prueba {X_pref_test_emb.shape}")

TF-IDF     -> entrenamiento (134, 6092), prueba (34, 6092)
Embeddings -> entrenamiento (134, 512), prueba (34, 512)


# PASO 6
## Comprobar las predicciones

In [59]:
from sklearn.metrics import classification_report, precision_recall_fscore_support

for nombre, pred in [("Modelo A (TF-IDF)", pred_a), ("Modelo B (embeddings)", pred_b)]:
    print(f"===== {nombre} =====")
    print(classification_report(y_pref_test, pred, zero_division=0))
    print("Matriz de confusión (filas = real 0/1, columnas = predicho 0/1):")
    print(confusion_matrix(y_pref_test, pred, labels=[0, 1]))
    print()

===== Modelo A (TF-IDF) =====
              precision    recall  f1-score   support

           0       0.82      0.96      0.89        28
           1       0.00      0.00      0.00         6

    accuracy                           0.79        34
   macro avg       0.41      0.48      0.44        34
weighted avg       0.67      0.79      0.73        34

Matriz de confusión (filas = real 0/1, columnas = predicho 0/1):
[[27  1]
 [ 6  0]]

===== Modelo B (embeddings) =====
              precision    recall  f1-score   support

           0       1.00      0.86      0.92        28
           1       0.60      1.00      0.75         6

    accuracy                           0.88        34
   macro avg       0.80      0.93      0.84        34
weighted avg       0.93      0.88      0.89        34

Matriz de confusión (filas = real 0/1, columnas = predicho 0/1):
[[24  4]
 [ 0  6]]



In [60]:
filas = []
for nombre, pred in [("A: TF-IDF", pred_a), ("B: embeddings", pred_b)]:
    p, r, f, _ = precision_recall_fscore_support(y_pref_test, pred, labels=[1], zero_division=0)
    filas.append({
        "modelo": nombre,
        "precisión (1)": round(p[0], 2),
        "recall (1)": round(r[0], 2),
        "F1 (1)": round(f[0], 2),
        "predicciones positivas": int(pred.sum()),
        "aciertos": int((pred == y_pref_test).sum()),
        "errores": int((pred != y_pref_test).sum()),
    })

display(pd.DataFrame(filas).set_index("modelo"))

,precisión (1),recall (1),F1 (1),predicciones positivas,aciertos,errores
modelo,,,,,,
A: TF-IDF,0.0,0.0,0.00,1,27,7
B: embeddings,0.6,1.0,0.75,10,30,4


#### Validación cruzada

Con 20 libros de prueba un solo libro cambia mucho las métricas. Para ver cuánto varían se repite una validación cruzada sobre los libros de entrenamiento, con la misma configuración. En el modelo A se usa un `Pipeline` para que el `TfidfVectorizer` se ajuste dentro de cada partición. No se usa para elegir parámetros.

In [61]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

pipeline_a = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(C=1, max_iter=1000, class_weight="balanced"))
])
f1_a = cross_val_score(pipeline_a, pref_entrenamiento["synopsis"], y_pref_train, cv=cv, scoring="f1")

f1_b = cross_val_score(
    LogisticRegression(C=1, max_iter=1000, class_weight="balanced"),
    X_pref_train_emb, y_pref_train, cv=cv, scoring="f1"
)

print(f"Modelo A (TF-IDF):     F1 clase 1 = {f1_a.mean():.2f} ± {f1_a.std():.2f}")
print(f"Modelo B (embeddings): F1 clase 1 = {f1_b.mean():.2f} ± {f1_b.std():.2f}")

Modelo A (TF-IDF):     F1 clase 1 = 0.02 ± 0.09
Modelo B (embeddings): F1 clase 1 = 0.51 ± 0.19


#### Dos libros de prueba

Se muestran dos libros de prueba con su valoración real y lo que predijo cada modelo; se elige al menos un error si lo hubo.

In [62]:
resultados_prueba = pref_prueba[["title", "interes"]].copy()
resultados_prueba["pred_A"] = pred_a
resultados_prueba["pred_B"] = pred_b

errores = resultados_prueba[(resultados_prueba["pred_A"] != resultados_prueba["interes"]) |
                            (resultados_prueba["pred_B"] != resultados_prueba["interes"])]
aciertos = resultados_prueba[(resultados_prueba["pred_A"] == resultados_prueba["interes"]) &
                             (resultados_prueba["pred_B"] == resultados_prueba["interes"])]

ejemplos = pd.concat([errores.head(1), aciertos.head(1)])
for slug, fila in ejemplos.iterrows():
    print(f"{fila['title']}: real {fila['interes']} | A predijo {fila['pred_A']} | B predijo {fila['pred_B']}")
    print(pref_prueba.loc[slug, "synopsis"][:400], "...\n")

Cómo explicar física cuántica con un gato zombi: real 1 | A predijo 0 | B predijo 1
¿Sabías que el teletransporte es real? ¿Que a veces un electrón tiene probabilidades de atravesar una pared? ¿Que dos partículas pueden influenciarse mutuamente aunque estén a años luz de distancia? ¿Y que las partículas cuánticas son como Clark Kent y disimulan sus poderes cuando los científicos las están observando? En Cómo explicar la física cuántica con un gato zombi descubrirás que, aunque no ...

Palestina en pedazos: real 0 | A predijo 0 | B predijo 0
Palestina en pedazos amplía y profundiza la reflexión que Lina Meruane inició en 2012, tras un viaje a Beit Jala que constituiría un paradójico volver a casa en nombre de quienes nunca pudieron regresar. Desde esa posición acaso desautorizada pero políticamente comprometida, Meruane escribió la crónica que abre este volumen -"Volverse Palestina"- y continuó escribiendo porque, como diría después,  ...



**Qué pasó con cada libro**

- ***Cómo explicar física cuántica con un gato zombi*** (interes real 1): el modelo B lo acierta y el modelo A da un **falso negativo**, porque lo predice 0 aunque al lector le interesaba. Es divulgación científica con humor, justo el tipo de libro que el lector valoró con 1. Los embeddings capturan ese sentido general («ciencia contada de forma entretenida») aunque las palabras exactas no se repitan en los libros de entrenamiento.
- ***Palestina en pedazos*** (interes real 0): los dos modelos aciertan con 0. Es un ensayo político, un tipo de libro que el lector rechazó varias veces.

**Comparación general.** El modelo A (TF-IDF) casi no predice positivos: una sola predicción de 1 en 34 libros, y fue un error. Con 134 libros de entrenamiento y más de 6000 palabras distintas, cada palabra aparece en muy pocas sinopsis y el modelo no logra separar las clases; termina prediciendo casi siempre la clase mayoritaria. El modelo B (embeddings) detecta los 6 libros de interés (recall 1.00) a cambio de 4 falsos positivos (precisión 0.60).

La validación cruzada confirma la diferencia: F1 de la clase 1 de 0.02 ± 0.09 para TF-IDF y 0.51 ± 0.19 para embeddings. El desvío alto muestra que, con tan pocos libros, las métricas de prueba son orientativas.

# PASO 7
## Convertir las predicciones en recomendaciones

Se ordenan los libros de prueba con tres criterios: la probabilidad de `interes=1` de cada clasificador y la similitud del coseno con un perfil del lector (el promedio de los embeddings de los libros de entrenamiento con `interes=1`).

In [63]:
from sklearn.metrics.pairwise import cosine_similarity

# Columna de predict_proba que corresponde a la clase 1
print("classes_ A:", modelo_a.classes_, "| classes_ B:", modelo_b.classes_)
col_a = list(modelo_a.classes_).index(1)
col_b = list(modelo_b.classes_).index(1)

ranking = pref_prueba[["title", "interes"]].copy()
ranking["proba_A"] = modelo_a.predict_proba(X_pref_test_tfidf)[:, col_a]
ranking["proba_B"] = modelo_b.predict_proba(X_pref_test_emb)[:, col_b]

# Perfil del lector: solo con los positivos de entrenamiento
perfil = X_pref_train_emb[y_pref_train.values == 1].mean(axis=0, keepdims=True)
ranking["similitud_perfil"] = cosine_similarity(perfil, X_pref_test_emb)[0]

classes_ A: [0 1] | classes_ B: [0 1]


In [64]:
listas = {
    "TF-IDF + clasificador": "proba_A",
    "Embeddings + clasificador": "proba_B",
    "Similitud con el perfil": "similitud_perfil",
}

precision_a_5 = {}
for nombre, columna in listas.items():
    top5 = ranking.sort_values(columna, ascending=False).head(5)
    precision_a_5[nombre] = top5["interes"].sum() / 5
    print(f"===== {nombre}: precision@5 = {precision_a_5[nombre]:.2f} =====")
    display(top5[["title", columna, "interes"]])

azar = y_pref_test.mean()
print(f"Proporción de 1 en prueba (precisión esperada al azar): {azar:.2f}")
print(f"Positivos en prueba: {y_pref_test.sum()} (si son menos de 5, ninguna lista puede llegar a 1)")

===== TF-IDF + clasificador: precision@5 = 0.60 =====


,title,proba_A,interes
2,La teoría de la evolución,0.503281,0
6,Los matemáticos serios son los que no se ríen ...,0.493454,1
161,Un viaje optimista por el futuro,0.484313,1
33,Cómo explicar física cuántica con un gato zombi,0.434965,1
150,Roger Waters,0.425038,0


===== Embeddings + clasificador: precision@5 = 0.80 =====


,title,proba_B,interes
33,Cómo explicar física cuántica con un gato zombi,0.748535,1
2,La teoría de la evolución,0.735156,0
161,Un viaje optimista por el futuro,0.721425,1
22,Steve Jobs & Apple,0.657847,1
100,La caza de hackers,0.638214,1


===== Similitud con el perfil: precision@5 = 0.40 =====


,title,similitud_perfil,interes
100,La caza de hackers,0.549409,1
22,Steve Jobs & Apple,0.515079,1
30,Amistades literarias,0.510290,0
192,En defensa de las vacunas,0.506411,0
137,Cordura y locura en Cervantes,0.477786,0


Proporción de 1 en prueba (precisión esperada al azar): 0.18
Positivos en prueba: 6 (si son menos de 5, ninguna lista puede llegar a 1)


In [65]:
display(pd.Series(precision_a_5, name="precision@5").to_frame().assign(azar=round(azar, 2)))

,precision@5,azar
TF-IDF + clasificador,0.6,0.18
Embeddings + clasificador,0.8,0.18
Similitud con el perfil,0.4,0.18


**¿Los clasificadores recomendaron mejor que la similitud?** Sí. Con 6 positivos en 34 libros de prueba, elegir al azar acierta un 18 %.

| Lista | precision@5 | Aciertos |
|---|---|---|
| Embeddings + clasificador | 0.80 | 4 de 5 |
| TF-IDF + clasificador | 0.60 | 3 de 5 |
| Similitud con el perfil | 0.40 | 2 de 5 |

Las tres listas superan claramente al azar, y las de los clasificadores superan a la similitud.

- **TF-IDF:** aunque casi nunca supera el umbral de 0.5 para predecir 1, al **ordenar** por probabilidad igual pone arriba varios libros de interés. Para recomendar alcanza con ordenar bien.
- **Similitud con el perfil:** promedia los embeddings de todos los positivos, que son muy variados (ciencia, filosofía estoica, tecnología, thrillers). El promedio queda en un punto intermedio que se parece a muchos ensayos. Por eso entran *Amistades literarias* o *Cordura y locura en Cervantes*, que el lector rechazó.
- **Clasificador:** aprende también de los negativos qué tipo de libro **no** le interesa.

El único error de la lista de embeddings es *La teoría de la evolución*: es divulgación científica, como varios positivos, pero en formato de texto de referencia, y el lector la valoró con 0.

# PASO 8
## Recomendar libros del catálogo completo

Se usa el clasificador con embeddings del paso 5, sin volver a ajustarlo, sobre los libros del catálogo limpio que el lector no vio (se excluyen todos los de `preferencias.csv`, incluidas las abstenciones).

In [66]:
vistos = set(preferencias["slug"])
candidatos = catalogo_limpio[~catalogo_limpio["slug"].isin(vistos)].copy()

# Otra versión de un libro ya valorado: mismo autor y título casi igual
valorados = catalogo_limpio[catalogo_limpio["slug"].isin(vistos)]
titulos_por_autor = valorados.groupby("authors_norm")["title_norm"].apply(list).to_dict()

def es_otra_version(fila):
    return any(
        fuzz.token_set_ratio(fila["title_norm"], t) >= 90
        for t in titulos_por_autor.get(fila["authors_norm"], [])
    )

otra_version = candidatos.apply(es_otra_version, axis=1)
print(f"Candidatos excluidos por ser otra versión de un libro valorado: {otra_version.sum()}")
display(candidatos.loc[otra_version, ["title", "authors_norm"]])
candidatos = candidatos[~otra_version]
print(f"Candidatos: {len(candidatos)}")

Candidatos excluidos por ser otra versión de un libro valorado: 5


,title,authors_norm
1464,Conversaciones Con Dios II,Neale Donald Walsch
1466,Conversaciones Con Dios III,Neale Donald Walsch
3741,Torres en la cocina 3,Sergio y Javier Torres
3746,Torres en la cocina,Sergio y Javier Torres
4301,Tirano Banderas,Ramón María del Valle-Inclán


Candidatos: 8083


In [67]:
X_candidatos = modelo_emb.encode(candidatos["synopsis"].tolist(), batch_size=64)
candidatos["puntaje"] = modelo_b.predict_proba(X_candidatos)[:, col_b]

top_catalogo = candidatos.sort_values("puntaje", ascending=False).head(5)
for _, fila in top_catalogo.iterrows():
    print(f"[{fila['puntaje']:.3f}] {fila['title']} ({fila['authors_norm']})")
    print(fila["synopsis"], "\n")

[0.873] Física de lo imposible (Michio Kaku)
Teletransporte, máquinas del tiempo, campos de fuerza y naves intergalácticas: ¿materia de ciencia ficción o tecnologías potencialmente disponibles en el futuro? Inspirado por los mundos fantásticos de H.G. Wells, Star Trek o La guerra de las galaxias, el prestigioso físico teórico y divulgador Michio Kaku lanza una mirada inteligente, seria y a menudo sorprendente a lo que nuestra actual comprensión de las leyes del universo nos permite vislumbrar en el futuro. Un fascinante viaje por mundos hasta ahora solo soñados por la ciencia ficción, que ofrece nuevos retos a las próximas generaciones de científicos capaces de desafiar las imposibilidades de nuestro tiempo. 

[0.838] Los viajes en el tiempo y el universo de Einstein (J. Richard Gott)
En el universo newtoniano, el viaje en el tiempo era una fantasía inconcebible. Sin embargo, en determinadas condiciones, en el universo de Einstein esta paradoja puede hacerse realidad. J. Richard Gott, 

**Comentario del lector:**
- **Me interesan:** *Física de lo imposible* (Michio Kaku) y *Los viajes en el tiempo y el universo de Einstein*. Mezclan ciencia real con ideas de ciencia ficción, que es lo que más me atrae.
- **Podría ser:** *El universo elegante*, aunque puede ser más pesado de lo que busco para leer por placer.
- **No me interesan:** *El universo holográfico* (suena a pseudociencia) y *Richard Dawkins contra Stephen Jay Gould* (un debate académico sobre biología evolutiva).

En general el modelo acertó el tema (ciencia y universo), pero las cinco son de divulgación. No aparece ninguna novela de ciencia ficción o thriller con trama, que es lo que más me gusta para descansar. Este comentario es una observación sobre candidatos nuevos y se mantiene separado de las métricas de prueba: una puntuación alta es una estimación del modelo, no una garantía de que el libro le vaya a gustar.

#### Resumir las recomendaciones

Resumen extractivo de dos oraciones: se divide la sinopsis en oraciones, se representan con TF-IDF, se calcula la similitud del coseno entre ellas y se conservan las dos más centrales (mayor similitud total con las demás) en su orden original.

In [68]:
def resumir(texto, n=2):
    ors = oraciones(texto)
    if len(ors) <= n:
        return " ".join(ors) if ors else texto
    X_or = TfidfVectorizer().fit_transform(ors)
    centralidad = cosine_similarity(X_or).sum(axis=1)
    elegidas = sorted(np.argsort(centralidad)[::-1][:n])
    return " ".join(ors[i] for i in elegidas)

for _, fila in top_catalogo.iterrows():
    print(f"{fila['title']}")
    print(f"  Resumen: {resumir(fila['synopsis'])}\n")

Física de lo imposible
  Resumen: Teletransporte, máquinas del tiempo, campos de fuerza y naves intergalácticas: ¿materia de ciencia ficción o tecnologías potencialmente disponibles en el futuro? Un fascinante viaje por mundos hasta ahora solo soñados por la ciencia ficción, que ofrece nuevos retos a las próximas generaciones de científicos capaces de desafiar las imposibilidades de nuestro tiempo.

Los viajes en el tiempo y el universo de Einstein
  Resumen: Richard Gott, astrofísico de la Universidad de Princeton y uno de los más destacados investigadores de esta materia, nos acompaña en un fascinante periplo hasta los límites más sorprendentes de la imaginación y la ciencia: ¿qué haríamos si dispusiéramos de una máquina del tiempo?, ¿nos lanzaríamos a un recorrido turístico por los siglos futuros?, ¿podríamos regresar al pasado y alterar el curso de la historia? Pero quizá los más sorprendente de este libro, tan apasionante como riguroso, radica en el hecho de que el estudio de los 

**Un caso que funcionó y uno que no**

- **Funcionó bien: *Física de lo imposible*.** Conservó la primera oración, que plantea el tema (teletransporte, máquinas del tiempo, ¿ciencia ficción o tecnología futura?), y la última, que dice qué ofrece el libro. Con dos oraciones se entiende de qué trata.
- **Perdió información importante: *El universo holográfico*.** Eligió dos oraciones del medio y del final, que hablan de percepción y de prejuicios de la ciencia. Dejó afuera la idea central del libro, que estaba en la segunda oración: «el universo es un gigantesco holograma». Las oraciones más «centrales» según TF-IDF son las que comparten más palabras con el resto, no necesariamente las que presentan la tesis.

#### Describir el perfil del lector

Con spaCy se extraen las frases sustantivas (`noun_chunks`) y las entidades nombradas de las sinopsis de entrenamiento, separadas por `interes`. Las frases se agrupan por el lema de su núcleo.

In [69]:
import spacy
from collections import Counter

nlp = spacy.load("es_core_news_lg")

def perfil_textos(textos):
    frases, entidades = Counter(), Counter()
    for doc in nlp.pipe(textos):
        frases.update(ch.root.lemma_.lower() for ch in doc.noun_chunks if not ch.root.is_stop)
        entidades.update(ent.text for ent in doc.ents)
    return frases, entidades

for valor in [1, 0]:
    frases, entidades = perfil_textos(pref_entrenamiento.loc[pref_entrenamiento["interes"] == valor, "synopsis"])
    print(f"===== interes = {valor} =====")
    display(pd.DataFrame({
        "frase sustantiva (lema del núcleo)": [f"{k} ({v})" for k, v in frases.most_common(10)],
        "entidad": [f"{k} ({v})" for k, v in entidades.most_common(10)],
    }))

===== interes = 1 =====


,frase sustantiva (lema del núcleo),entidad
0,vida (15),Epicteto (4)
1,libro (14),Flavio Arriano (3)
2,año (11),Universo (3)
3,ciencia (11),Nicópolis (2)
4,universo (11),Sin embargo (2)
5,tiempo (8),Discursos (2)
6,mundo (7),Roma (2)
7,historia (6),Feynman (2)
8,posibilidad (5),Wells (2)
9,problema (5),Bea (2)


===== interes = 0 =====


,frase sustantiva (lema del núcleo),entidad
0,libro (61),España (6)
1,vida (56),Hollywood (6)
2,obra (40),Londres (5)
3,mundo (39),Este libro (5)
4,año (34),Nueva York (5)
5,historia (28),Aviraneta (5)
6,tiempo (25),Nixon (5)
7,autor (24),Bernanos (5)
8,novela (20),París (4)
9,guerra (18),Sarah (4)


**Comentario del lector:** En parte.
- **Lo que acierta:** para `interes = 1` aparecen *ciencia*, *universo*, *tiempo*, *posibilidad* y *problema*, y entidades como Feynman, Wells o Epicteto. Eso describe bien lo que me interesa: ciencia, ideas sobre el universo, resolver problemas y filosofía práctica como la estoica.
- **Lo que no sirve para distinguir:** palabras como *vida*, *libro* o *año*, que aparecen arriba en las dos listas.
- **Errores de spaCy:** marca como entidad «Sin embargo».
- **Lo que falta:** la ficción con trama, el humor y el deporte. Hubo pocos libros de ese tipo entre los valorados, así que el perfil queda sesgado hacia la divulgación científica.

La lista de `interes = 0` (*novela*, *obra*, *autor*, *guerra*, España, Hollywood) refleja bien lo que rechacé: crítica literaria, historia, teatro y novela realista.

#### Función final

In [70]:
def recomendar(n=5):
    """Devuelve los n libros del catálogo limpio no valorados con mayor probabilidad de interes=1
    según el clasificador con embeddings, con su resumen extractivo."""
    top = candidatos.sort_values("puntaje", ascending=False).head(n)
    return pd.DataFrame({
        "slug": top["slug"],
        "title": top["title"],
        "puntaje": top["puntaje"].round(3),
        "resumen": top["synopsis"].apply(resumir),
    }).reset_index(drop=True)

recomendar(5)

,slug,title,puntaje,resumen
0,fisica-de-lo-imposible,Física de lo imposible,0.873,"Teletransporte, máquinas del tiempo, campos de..."
1,los-viajes-en-el-tiempo-y-el-universo-de-einstein,Los viajes en el tiempo y el universo de Einstein,0.838,"Richard Gott, astrofísico de la Universidad de..."
2,el-universo-holografico,El universo holográfico,0.838,"De esta manera, el tiempo y el espacio no son ..."
3,richard-dawkins-contra-stephen-jay-gould,Richard Dawkins contra Stephen Jay Gould,0.837,La historia de la ciencia está repleta de riva...
4,el-universo-elegante,El universo elegante,0.836,Brian Greene explica en este libro cómo las gr...
